# Check Unity Catalog Layer

Lists current Unity Catalog metadata for one selected Ampere layer without changing catalog state.

Steps performed:
1. Select layer and contract path.
2. Load `.env`, UC runtime config, and canonical table contract through `tools/py_utils`.
3. Resolve the schemas assigned to the selected layer.
4. Query Unity Catalog for registered tables in each schema.
5. Print format, column count, and storage location for quick validation.


In [1]:
# -----------------------------------------------------------------------------
# Manual setup
# -----------------------------------------------------------------------------
LAYER = "gold"  # bronze | silver | gold
CONTRACT_PATH = "tools/uc/contracts/ampere_tables.json"
VERBOSE = True


In [2]:
# -----------------------------------------------------------------------------
# Imports and project path setup
# -----------------------------------------------------------------------------
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find project root from current notebook path.")

from tools.py_utils.uc_client import UCClient  # noqa: E402
from tools.py_utils.uc_config import load_uc_bulk_config, uc_client_config  # noqa: E402
from tools.py_utils.uc_contracts import layer_schemas, layers, load_contract  # noqa: E402


In [3]:
# -----------------------------------------------------------------------------
# Read UC metadata for the selected layer
# -----------------------------------------------------------------------------
contract = load_contract(CONTRACT_PATH)
contract_layers = layers(contract)
if LAYER not in contract_layers:
    raise ValueError(f"Unknown layer {LAYER!r}. Available: {sorted(contract_layers)}")

config = load_uc_bulk_config()
uc = UCClient(uc_client_config(config, verbose=VERBOSE))
registered_tables = []
for schema in layer_schemas(contract, LAYER):
    schema_name = schema["name"]
    tables = uc.list_tables(schema_name)
    print(f"Tables in {uc.config.catalog}.{schema_name}: {len(tables)}")
    for table in tables:
        print(
            f"  - {table['name']:<35} "
            f"{table.get('table_type', '?'):<10} "
            f"{table.get('data_source_format', '?'):<8} "
            f"cols={len(table.get('columns', [])):<3} "
            f"{table.get('storage_location', '')}"
        )
    registered_tables.extend(tables)

registered_tables


[debug] remote cmd: kubectl -n unity-catalog get endpoints unity-catalog-unitycatalog-server --no-headers | awk '{print $2}' | cut -d: -f1
[debug] stdout: 10.10.4.48
[debug] remote cmd: curl -sS --connect-timeout 10 --max-time 60 -X GET http://10.10.4.48:8080/api/2.1/unity-catalog/tables -G --data-urlencode catalog_name=ampere --data-urlencode schema_name=gold
[debug] stdout: {"tables":[{"name":"budget_orders_sales","catalog_name":"ampere","schema_name":"gold","table_type":"EXTERNAL","data_source_format":"DELTA","columns":[{"name":"budget_name","type_text":"string","type_json":"{\"name\": \"string\"}","type_name":"STRING","type_precision":0,"type_scale":0,"type_interval_type":null,"position":1,"comment":null,"nullable":false,"partition_index":null},{"name":"budget_date","type_text":"date","type_json":"{\"name\": \"date\"}","type_name":"DATE","type_precision":0,"type_scale":0,"type_interval_type":null,"position":2,"comment":null,"nullable":false,"partition_index":null},{"name":"store_id

[{'name': 'budget_orders_sales',
  'catalog_name': 'ampere',
  'schema_name': 'gold',
  'table_type': 'EXTERNAL',
  'data_source_format': 'DELTA',
  'columns': [{'name': 'budget_name',
    'type_text': 'string',
    'type_json': '{"name": "string"}',
    'type_name': 'STRING',
    'type_precision': 0,
    'type_scale': 0,
    'type_interval_type': None,
    'position': 1,
    'comment': None,
    'nullable': False,
    'partition_index': None},
   {'name': 'budget_date',
    'type_text': 'date',
    'type_json': '{"name": "date"}',
    'type_name': 'DATE',
    'type_precision': 0,
    'type_scale': 0,
    'type_interval_type': None,
    'position': 2,
    'comment': None,
    'nullable': False,
    'partition_index': None},
   {'name': 'store_id',
    'type_text': 'smallint',
    'type_json': '{"name": "short"}',
    'type_name': 'SHORT',
    'type_precision': 0,
    'type_scale': 0,
    'type_interval_type': None,
    'position': 3,
    'comment': None,
    'nullable': False,
    'par